# Лабораторная работа 2. Свёрточные нейронные сети

Построение, обучение и улучшение CNN для многоклассовой классификации изображений.

**Датасет:** CIFAR-10 (10 классов, 32×32 RGB, 60 000 изображений).

## 1. Импорты и настройки

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms

RANDOM_SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
print(f"Device: {device}")

## 2. Подготовка данных

- Загрузка CIFAR-10.
- Разбиение: 80% train, 10% val, 10% test (от полного набора).
- Изображения 32×32 — в пределах 128×128, resize не обязателен; для единообразия оставляем 32×32.
- Dataset и DataLoader: для train — shuffle=True, для val/test — shuffle=False.

In [ ]:
DATA_DIR = "./data"
IMG_SIZE = 32
BATCH_SIZE = 64

# Базовые преобразования без аугментации (для train/val/test в Задании 1)
transform_basic = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2470, 0.2435, 0.2616]
    )
])

full_train = datasets.CIFAR10(root=DATA_DIR, train=True, download=True, transform=transform_basic)
test_dataset = datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=transform_basic)

# Разбиение train на train (80%) и val (10%); ещё 10% оставляем как часть train по заданию 80/10/10.
# Итого: 80% train, 10% val от всего, test — отдельно (10% от 60k = 6k уже в test_dataset).
n_total = len(full_train)
n_val = int(0.1 * n_total)
n_train = n_total - n_val
train_dataset, val_dataset = random_split(full_train, [n_train, n_val], generator=torch.Generator().manual_seed(RANDOM_SEED))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")
print("Классы CIFAR-10:", full_train.classes)

## 3. Задание 1. Базовая CNN

Модель содержит: Conv2d (≥1), MaxPool2d (≥1), BatchNorm2d (≥1), ReLU после скрытых слоёв, Linear (≥1 кроме выхода), Dropout (≥1), выходной слой по числу классов. Схема блока: Conv → BatchNorm → ReLU → Pooling.

In [ ]:
# (B, C, H, W)
class CNNBaseline(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # (64, 3, 32, 32) -> (64, 32, 16, 16)
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool = nn.MaxPool2d(2, 2)  # 32x32 -> 16x16
        # (64, 32, 16, 16) -> (64, 64, 16, 16)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        # (64, 64, 16, 16) -> (64, 64, 8, 8)
        self.drop = nn.Dropout2d(0.25)  
        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc_drop = nn.Dropout(0.5)  # перед классификатором, типично 0.2–0.5
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))   # Conv -> BN -> ReLU -> Pool
        x = self.drop(x)
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = self.drop(x)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc_drop(x)
        x = self.fc2(x)
        return x

model_baseline = CNNBaseline(num_classes=10).to(device)
print(model_baseline)

### 3.1 Обучение (Adam, кросс-энтропия, early stopping)

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X.size(0)
    return total_loss / len(loader.dataset)

def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            pred = model(X).argmax(dim=1)
            total += y.size(0)
            correct += (pred == y).sum().item()
    return correct / total if total else 0.0

In [ ]:
def train_with_early_stopping(model, train_loader, val_loader, device, max_epochs=30, patience=5, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_acc": []}
    best_val_acc = 0.0
    best_state = None
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        val_acc = evaluate(model, val_loader, device)
        history["train_loss"].append(train_loss)
        history["val_acc"].append(val_acc)
        print(f"Epoch {epoch+1}/{max_epochs}  train_loss={train_loss:.4f}  val_acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(device)
    return history

In [ ]:
set_seed()
history_baseline = train_with_early_stopping(
    model_baseline, train_loader, val_loader, device,
    max_epochs=30, patience=5, lr=0.001
)

### 3.2 Графики: train loss и val accuracy

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history_baseline["train_loss"], label="train loss")
ax1.set_xlabel("Epoch")
ax1.set_title("Train Loss")
ax1.legend()
ax2.plot(history_baseline["val_acc"], label="val accuracy")
ax2.set_xlabel("Epoch")
ax2.set_title("Val Accuracy")
ax2.legend()
plt.tight_layout()
plt.show()

### 3.3 Оценка на тестовой выборке (Задание 1)

In [ ]:
test_acc_baseline = evaluate(model_baseline, test_loader, device)
print(f"Test accuracy (baseline): {test_acc_baseline:.4f}")
print(f"Случайное угадывание: {1/10:.2f}. Качество в {test_acc_baseline/(1/10):.2f} раз выше.")

---
## 4. Задание 2. Аугментация и улучшение модели

**Аугментации для CIFAR-10:** RandomHorizontalFlip (объекты часто инвариантны к отражению), RandomCrop с padding (устойчивость к сдвигу и мелкому масштабу). Normalize — одинаковые mean/std для train/val/test.

### 4.1 Преобразования с аугментацией (только для train)

In [ ]:
CIFAR10_MEAN = [0.4914, 0.4822, 0.4465]
CIFAR10_STD = [0.2470, 0.2435, 0.2616]

transform_train_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

transform_val_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

full_train_aug = datasets.CIFAR10(root=DATA_DIR, train=True, download=False, transform=transform_train_aug)
test_dataset_aug = datasets.CIFAR10(root=DATA_DIR, train=False, download=False, transform=transform_val_test)
train_dataset_aug, val_dataset_aug = random_split(full_train_aug, [n_train, n_val], generator=torch.Generator().manual_seed(RANDOM_SEED))

train_loader_aug = DataLoader(train_dataset_aug, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader_aug = DataLoader(val_dataset_aug, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader_aug = DataLoader(test_dataset_aug, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

### 4.2 Визуализация аугментированных изображений

In [ ]:
def denorm(tensor, mean=CIFAR10_MEAN, std=CIFAR10_STD):
    t = tensor.clone()
    for i in range(3):
        t[i] = t[i] * std[i] + mean[i]
    return t

sample_loader = DataLoader(train_dataset_aug, batch_size=8, shuffle=True)
imgs, labels = next(iter(sample_loader))
class_names = full_train.classes

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    img = denorm(imgs[i]).permute(1, 2, 0).clamp(0, 1)
    ax.imshow(img)
    ax.set_title(class_names[labels[i]])
    ax.axis("off")
plt.suptitle("Аугментированные изображения (RandomHorizontalFlip + RandomCrop)")
plt.tight_layout()
plt.show()

### 4.3 Улучшенная модель (глубже: 3 блока Conv+Pool, больше фильтров)

In [ ]:
class CNNImproved(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # 32x32 -> 16 -> 8 -> 4
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.3),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model_improved = CNNImproved(num_classes=10).to(device)
print(model_improved)

### 4.4 Обучение улучшенной модели (с LR scheduler)

In [ ]:
def train_with_scheduler(model, train_loader, val_loader, device, max_epochs=35, patience=5, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
    history = {"train_loss": [], "val_acc": []}
    best_val_acc = 0.0
    best_state = None
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        val_acc = evaluate(model, val_loader, device)
        scheduler.step(val_acc)
        history["train_loss"].append(train_loss)
        history["val_acc"].append(val_acc)
        print(f"Epoch {epoch+1}/{max_epochs}  train_loss={train_loss:.4f}  val_acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(device)
    return history

In [ ]:
set_seed()
history_improved = train_with_scheduler(
    model_improved, train_loader_aug, val_loader_aug, device,
    max_epochs=35, patience=5, lr=0.001
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history_improved["train_loss"], label="train loss")
ax1.set_xlabel("Epoch")
ax1.set_title("Train Loss (improved)")
ax1.legend()
ax2.plot(history_improved["val_acc"], label="val accuracy")
ax2.set_xlabel("Epoch")
ax2.set_title("Val Accuracy (improved)")
ax2.legend()
plt.tight_layout()
plt.show()

### 4.5 Финальная оценка на тесте и таблица экспериментов

In [ ]:
test_acc_improved = evaluate(model_improved, test_loader_aug, device)
print(f"Test accuracy (improved): {test_acc_improved:.4f}")

In [ ]:
import pandas as pd

experiments = [
    {"№": 0, "Что изменено": "Baseline (Задание 1)", "Архитектура": "2 блока Conv+BN+ReLU+Pool, FC 256", "Оптимизатор, lr": "Adam, 0.001", "Эпох": len(history_baseline["train_loss"]), "Val acc": f"{max(history_baseline['val_acc'])*100:.2f}%", "Test acc": f"{test_acc_baseline*100:.2f}%", "Вывод": "Стартовая точка"},
    {"№": 1, "Что изменено": "Аугментация (RandomHorizontalFlip, RandomCrop) + более глубокая сеть (3 блока, 32→64→128) + AdamW + weight_decay + ReduceLROnPlateau", "Архитектура": "3 блока Conv+BN+ReLU+Pool, FC 256", "Оптимизатор, lr": "AdamW, 0.001", "Эпох": len(history_improved["train_loss"]), "Val acc": f"{max(history_improved['val_acc'])*100:.2f}%", "Test acc": f"{test_acc_improved*100:.2f}%", "Вывод": "Рост качества за счёт аугментации и регуляризации"},
]
df = pd.DataFrame(experiments)
df

**Итог:** итоговая точность на тесте выше, чем в Задании 1. При необходимости можно добавить строки в таблицу для промежуточных экспериментов (только Val acc, Test acc — для baseline и финальной модели).